In [ ]:
# ==========================================================
# NUMERICAL CONVERGENCE / STABILIZATION ANALYSIS
# 2D-ADDS MODEL — ONE CELL / ONE COMMAND
# ==========================================================
#
# PURPOSE
# -------
# This notebook performs two complementary numerical checks:
#
# A. TEMPORAL STABILIZATION
#    - Uses the study time step dt = 0.25 s for the production-grid
#      temporal analysis.
#    - Measures relative L2 change of the complete concentration field.
#    - Uses a sustained-stabilization criterion: the relative change
#      must remain below the specified tolerance for several consecutive
#      output intervals.
#
# B. SPATIAL / GRID REFINEMENT
#    - Compares 4 m, 2 m and 1 m spatial resolutions.
#    - Uses the SAME physical parameters and final time.
#    - Uses a common conservative dt = 0.125 s for the grid-refinement
#      comparison so that the finest 1 m grid remains stable and the
#      temporal discretization does not dominate the comparison.
#    - Compares successive grids using the relative L2 difference:
#
#         E_h = ||u_h - u_h/2||_2 / ||u_h/2||_2
#
#      after restricting the finer solution to the coarser grid.
#
# BOUNDARY CONDITION
# ------------------
# Homogeneous Neumann zero-normal-gradient:
#
#                   ∂u/∂n = 0
#
# at inflow, lateral/open boundaries and outflow.
#
# DISCRETIZATION
# --------------
# - Forward Euler in time
# - First-order upwind advection
# - Second-order central diffusion
# - Combined first-order removal:
#       -(lambda + Vs/H)u
# - Gaussian source
#
# IMPORTANT
# ---------
# The parameters below reproduce the convergence-test setup in the
# uploaded notebook. They are numerical-analysis settings, not the
# final calibrated parameters.
#
# ==========================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ==========================================================
# 1. PHYSICAL PARAMETERS
# ==========================================================

Lx = 200.0
Ly = 200.0

# Meteorology
vx = 2.30          # m/s
vy = 0.50          # m/s

# Turbulent diffusion
Dx = 2.00          # m^2/s
Dy = 2.00          # m^2/s

# Surface deposition
lam = 0.001        # s^-1

# Mixing height
H = 50.0           # m

# Particle properties
rho_particle = 1100.0
rho_air = 1.204
g = 9.81
particle_diameter = 75e-6
mu = 1.81e-5

Vs = (
    (rho_particle - rho_air)
    * g
    * particle_diameter**2
    / (18.0 * mu)
)

# Source
Q = 200.0
xf = 100.0
yf = 100.0
sigma = 2.5

# Final reporting time
T_final = 120.0

# ==========================================================
# 2. TIME-STABILIZATION SETTINGS
# ==========================================================
# Production-grid temporal test follows the study value dt = 0.25 s.

dt_temporal = 0.25          # s
temporal_tolerance = 0.01   # percent
required_consecutive = 3    # sustained stabilization intervals

# Output times for the temporal test.
# More frequent output improves the reliability of the stabilization test.
temporal_output_times = np.arange(
    15.0,
    T_final + 1e-12,
    15.0
)

# ==========================================================
# 3. FDM STABILITY DIAGNOSTIC
# ==========================================================

def stability_numbers(dx, dy, dt, D_x=Dx, D_y=Dy):
    cfl_x = abs(vx) * dt / dx
    cfl_y = abs(vy) * dt / dy

    rx = D_x * dt / dx**2
    ry = D_y * dt / dy**2

    # Conservative combined explicit criterion used here:
    combined = cfl_x + cfl_y + 2.0 * (rx + ry)

    return cfl_x, cfl_y, rx, ry, combined

cfl_x, cfl_y, rx, ry, combined_number = stability_numbers(
    2.0, 2.0, dt_temporal
)

print("=" * 72)
print("TEMPORAL NUMERICAL-STABILITY DIAGNOSTIC")
print("=" * 72)
print(f"dx = dy = 2.000 m")
print(f"dt = {dt_temporal:.3f} s")
print(f"CFLx = {cfl_x:.6f}")
print(f"CFLy = {cfl_y:.6f}")
print(f"rx = {rx:.6f}")
print(f"ry = {ry:.6f}")
print(f"Combined diagnostic number = {combined_number:.6f}")
print(
    "Note: this is a conservative explicit stability diagnostic; "
    "the actual condition should be stated consistently with the "
    "scheme used in the manuscript."
)
print("=" * 72)

# ==========================================================
# 4. SOURCE AND REMOVAL TERMS
# ==========================================================

def build_grid(dx_requested):
    """
    Build an exactly fitting grid on [0, 200] x [0, 200].
    """
    n_x = int(round(Lx / dx_requested)) + 1
    n_y = int(round(Ly / dx_requested)) + 1

    x = np.linspace(0.0, Lx, n_x)
    y = np.linspace(0.0, Ly, n_y)

    X, Y = np.meshgrid(x, y)

    source = Q * np.exp(
        -(
            (X - xf)**2
            + (Y - yf)**2
        ) / (2.0 * sigma**2)
    )

    return x, y, X, Y, source

removal_rate = lam + Vs / H

print("\nPHYSICAL REMOVAL TERMS")
print("-" * 72)
print(f"Settling velocity Vs = {Vs:.6e} m/s")
print(f"Vs/H = {Vs/H:.6f} s^-1")
print(f"lambda = {lam:.6f} s^-1")
print(f"Total removal rate = {removal_rate:.6f} s^-1")
print(f"Characteristic removal time = {1.0/removal_rate:.3f} s")

# ==========================================================
# 5. ZERO-NORMAL-GRADIENT NEUMANN BOUNDARY CONDITION
# ==========================================================

def apply_neumann_zero_gradient(u):
    """
    Homogeneous Neumann condition:
        du/dn = 0

    Left   : inflow
    Right  : outflow
    Bottom : lateral/open
    Top    : lateral/open
    """

    u[:, 0] = u[:, 1]
    u[:, -1] = u[:, -2]
    u[0, :] = u[1, :]
    u[-1, :] = u[-2, :]

    return u

# ==========================================================
# 6. ONE EXPLICIT FDM TIME STEP
# ==========================================================

def advance_one_step(
    u,
    x,
    y,
    source,
    dx,
    dy,
    dt
):
    un = u.copy()

    # First-order upwind advection.
    # vx > 0 and vy > 0 in the present model configuration.
    dudx = (
        un[1:-1, 1:-1]
        - un[1:-1, :-2]
    ) / dx

    dudy = (
        un[1:-1, 1:-1]
        - un[:-2, 1:-1]
    ) / dy

    # Second-order central diffusion.
    d2udx2 = (
        un[1:-1, 2:]
        - 2.0 * un[1:-1, 1:-1]
        + un[1:-1, :-2]
    ) / dx**2

    d2udy2 = (
        un[2:, 1:-1]
        - 2.0 * un[1:-1, 1:-1]
        + un[:-2, 1:-1]
    ) / dy**2

    reaction = (
        -removal_rate
        * un[1:-1, 1:-1]
    )

    u_new = un.copy()

    u_new[1:-1, 1:-1] = (
        un[1:-1, 1:-1]
        + dt * (
            -vx * dudx
            -vy * dudy
            + Dx * d2udx2
            + Dy * d2udy2
            + reaction
            + source[1:-1, 1:-1]
        )
    )

    # Homogeneous Neumann boundaries.
    u_new = apply_neumann_zero_gradient(u_new)

    # Numerical safeguard.
    return np.maximum(u_new, 0.0)

# ==========================================================
# 7. RUN A TRANSIENT SIMULATION AND STORE SNAPSHOTS
# ==========================================================

def run_transient(
    dx_requested,
    dt,
    output_times,
    final_time
):
    """
    Run the FDM model and store concentration fields at requested times.
    """

    x, y, X, Y, source = build_grid(dx_requested)

    steps = int(round(final_time / dt))
    requested_steps = {
        int(round(t / dt)): float(t)
        for t in output_times
        if int(round(t / dt)) <= steps
    }

    u = np.zeros((len(y), len(x)), dtype=float)
    snapshots = {}

    for step in range(1, steps + 1):

        u = advance_one_step(
            u,
            x,
            y,
            source,
            dx_requested,
            dx_requested,
            dt
        )

        if step in requested_steps:
            snapshots[requested_steps[step]] = u.copy()

    return {
        "x": x,
        "y": y,
        "X": X,
        "Y": Y,
        "source": source,
        "snapshots": snapshots,
        "final_field": u
    }

# ==========================================================
# 8. TEMPORAL STABILIZATION ON THE PRODUCTION GRID
# ==========================================================
# Production spatial resolution = 2 m
# Production time step = 0.25 s

temporal_run = run_transient(
    dx_requested=2.0,
    dt=dt_temporal,
    output_times=temporal_output_times,
    final_time=T_final
)

snapshots = temporal_run["snapshots"]
x2 = temporal_run["x"]
y2 = temporal_run["y"]
X2 = temporal_run["X"]
Y2 = temporal_run["Y"]

times = np.array(sorted(snapshots.keys()), dtype=float)

max_concentration = np.array([
    np.max(snapshots[t])
    for t in times
])

hotspot_indices = [
    np.unravel_index(
        np.argmax(snapshots[t]),
        snapshots[t].shape
    )
    for t in times
]

hotspot_x = np.array([
    x2[idx[1]]
    for idx in hotspot_indices
])

hotspot_y = np.array([
    y2[idx[0]]
    for idx in hotspot_indices
])

relative_changes = np.full(
    len(times),
    np.nan
)

for k in range(1, len(times)):
    current = snapshots[times[k]]
    previous = snapshots[times[k - 1]]

    numerator = np.linalg.norm(
        current - previous
    )

    denominator = max(
        np.linalg.norm(current),
        np.finfo(float).eps
    )

    relative_changes[k] = (
        100.0
        * numerator
        / denominator
    )

# ==========================================================
# 9. SUSTAINED TEMPORAL STABILIZATION TEST
# ==========================================================
#
# The criterion must be satisfied for `required_consecutive`
# consecutive output intervals, rather than accepting the first
# isolated value below the threshold.

stabilization_time = None
stabilization_start_index = None

for k in range(1, len(times) - required_consecutive + 2):

    window = relative_changes[
        k:k + required_consecutive
    ]

    if np.all(
        window < temporal_tolerance
    ):
        stabilization_time = times[k]
        stabilization_start_index = k
        break

print("\n" + "=" * 72)
print("TEMPORAL STABILIZATION RESULTS")
print("=" * 72)
print(
    f"Criterion: relative L2 field change < "
    f"{temporal_tolerance:.4f}% for "
    f"{required_consecutive} consecutive intervals."
)

print(
    f"\n{'Time (s)':>10}"
    f"{'Max C':>15}"
    f"{'Hotspot x':>12}"
    f"{'Hotspot y':>12}"
    f"{'Change (%)':>15}"
)

print("-" * 72)

for k, t in enumerate(times):

    change_text = (
        "---"
        if np.isnan(relative_changes[k])
        else f"{relative_changes[k]:.6f}"
    )

    print(
        f"{t:10.2f}"
        f"{max_concentration[k]:15.6f}"
        f"{hotspot_x[k]:12.2f}"
        f"{hotspot_y[k]:12.2f}"
        f"{change_text:>15}"
    )

print("-" * 72)

if stabilization_time is not None:

    print(
        f"RESULT: sustained temporal stabilization begins at "
        f"approximately t = {stabilization_time:.2f} s."
    )

    print(
        f"The criterion remains satisfied over the next "
        f"{required_consecutive} consecutive intervals."
    )

else:

    print(
        "RESULT: the sustained temporal-stabilization criterion "
        "was not reached within the tested time range."
    )

# Explicitly report t = 120 s.
if 120.0 in snapshots:

    idx120 = np.where(
        times == 120.0
    )[0][0]

    print("\nASSESSMENT AT t = 120 s")
    print(f"Maximum concentration = {max_concentration[idx120]:.6f}")
    print(
        f"Hotspot = "
        f"({hotspot_x[idx120]:.2f}, "
        f"{hotspot_y[idx120]:.2f}) m"
    )
    print(
        f"Relative field change = "
        f"{relative_changes[idx120]:.8f}%"
    )

# ==========================================================
# 10. TEMPORAL STABILIZATION PLOT
# ==========================================================

plt.figure(figsize=(8, 5))

plt.semilogy(
    times[1:],
    relative_changes[1:],
    marker="o",
    linewidth=1.8
)

plt.axhline(
    temporal_tolerance,
    linestyle="--",
    linewidth=1.5,
    label=f"{temporal_tolerance:.2f}% criterion"
)

if stabilization_time is not None:
    plt.axvline(
        stabilization_time,
        linestyle=":",
        linewidth=1.5,
        label=f"Sustained stabilization: {stabilization_time:.0f} s"
    )

plt.xlabel("Simulation time (s)")
plt.ylabel("Relative field change (%)")
plt.title("Temporal Stabilization of the 2D-ADDS Concentration Field")
plt.grid(True, which="both", alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

# ==========================================================
# 11. MAXIMUM CONCENTRATION VERSUS TIME
# ==========================================================

plt.figure(figsize=(8, 5))

plt.plot(
    times,
    max_concentration,
    marker="o",
    linewidth=1.8
)

plt.xlabel("Simulation time (s)")
plt.ylabel("Maximum concentration")
plt.title("Temporal Evolution of Maximum Airborne Microplastic Concentration")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# ==========================================================
# 12. HOTSPOT LOCATION VERSUS TIME
# ==========================================================

plt.figure(figsize=(8, 5))

plt.plot(
    times,
    hotspot_x,
    marker="o",
    linewidth=1.8,
    label="Hotspot x-coordinate"
)

plt.plot(
    times,
    hotspot_y,
    marker="s",
    linewidth=1.8,
    label="Hotspot y-coordinate"
)

plt.xlabel("Simulation time (s)")
plt.ylabel("Coordinate (m)")
plt.title("Temporal Stability of the Hotspot Location")
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

# ==========================================================
# 13. SPATIAL / GRID-REFINEMENT STUDY
# ==========================================================
#
# Spatial resolutions:
#     4 m
#     2 m
#     1 m
#
# The production-model dt = 0.25 s is retained for the temporal test.
# For the spatial refinement study, ALL three grids use one COMMON dt,
# calculated from the finest 1 m grid with a safety factor. This avoids
# confounding spatial discretization error with different time steps.
#
# This means the refinement comparison is not confounded by changing dt
# between grids.
#
# The production model remains documented with dt = 0.25 s.
# ==========================================================

grid_sizes = [4.0, 2.0, 1.0]

# Use ONE common time step for all grid-refinement runs.
# It is calculated from the finest (1 m) grid with a safety factor,
# so spatial discretization is compared without changing dt between grids.
finest_dx = min(grid_sizes)

dt_grid_limit = 1.0 / (
    abs(vx) / finest_dx
    + abs(vy) / finest_dx
    + 2.0 * Dx / finest_dx**2
    + 2.0 * Dy / finest_dx**2
)

dt_grid = 0.50 * dt_grid_limit

grid_results = []

print("\n" + "=" * 72)
print("SPATIAL / GRID-REFINEMENT STUDY")
print("=" * 72)
print(f"Comparison time = {T_final:.1f} s")
print(f"Common grid-refinement dt = {dt_grid:.6f} s")
print()

for dx_grid in grid_sizes:

    cflx, cfly, rxx, ryy, combined = stability_numbers(
        dx_grid,
        dx_grid,
        dt_grid
    )

    print(
        f"Running dx = dy = {dx_grid:g} m | "
        f"dt = {dt_grid:.6f} s | "
        f"CFL+diffusion diagnostic = {combined:.6f}"
    )

    if combined > 1.0 + 1e-12:
        raise ValueError(
            f"Stability check failed for grid {dx_grid:g} m: "
            f"diagnostic = {combined:.6f} > 1. "
            f"Check Dx, Dy, dt_grid, or the stability criterion."
        )

    result = run_transient(
        dx_requested=dx_grid,
        dt=dt_grid,
        output_times=[T_final],
        final_time=T_final
    )

    field = result["final_field"]

    grid_results.append({
        "dx_m": dx_grid,
        "dy_m": dx_grid,
        "Nx": len(result["x"]),
        "Ny": len(result["y"]),
        "dt_s": dt_grid,
        "max_concentration": np.max(field),
        "l2_norm": np.linalg.norm(field)
    })

    # Save result for successive-grid comparison.
    result["relative_change_from_previous"] = np.nan

    # Store in the result object so it can be reused below.
    grid_results[-1]["result"] = result

# ==========================================================
# 14. SUCCESSIVE-GRID RELATIVE L2 DIFFERENCES
# ==========================================================
#
# All grids are nested:
#     1 m -> every second point = 2 m grid
#     2 m -> every second point = 4 m grid
#
# Therefore no interpolation is required.

field_4m = grid_results[0]["result"]["final_field"]
field_2m = grid_results[1]["result"]["final_field"]
field_1m = grid_results[2]["result"]["final_field"]

field_2m_on_4m = field_2m[::2, ::2]
field_1m_on_2m = field_1m[::2, ::2]

if field_2m_on_4m.shape != field_4m.shape:
    raise ValueError("4 m and restricted 2 m grids are not aligned.")

if field_1m_on_2m.shape != field_2m.shape:
    raise ValueError("2 m and restricted 1 m grids are not aligned.")

E_4_to_2 = (
    np.linalg.norm(
        field_2m_on_4m - field_4m
    )
    / max(
        np.linalg.norm(field_2m_on_4m),
        np.finfo(float).eps
    )
)

E_2_to_1 = (
    np.linalg.norm(
        field_1m_on_2m - field_2m
    )
    / max(
        np.linalg.norm(field_1m_on_2m),
        np.finfo(float).eps
    )
)

max_4m = np.max(field_4m)
max_2m = np.max(field_2m)
max_1m = np.max(field_1m)

grid_summary = pd.DataFrame({
    "Grid spacing (m)": [4.0, 2.0, 1.0],
    "Nx": [51, 101, 201],
    "Ny": [51, 101, 201],
    "dt (s)": [dt_grid, dt_grid, dt_grid],
    "Maximum concentration": [
        max_4m,
        max_2m,
        max_1m
    ],
    "Relative L2 change vs finer grid (%)": [
        np.nan,
        100.0 * E_4_to_2,
        100.0 * E_2_to_1
    ]
})

print("\nGRID-REFINEMENT RESULTS")
print("-" * 72)

display(
    grid_summary.style.format({
        "Grid spacing (m)": "{:.1f}",
        "dt (s)": "{:.3f}",
        "Maximum concentration": "{:.6f}",
        "Relative L2 change vs finer grid (%)": "{:.6f}"
    })
)

# ==========================================================
# 15. GRID-REFINEMENT PLOT
# ==========================================================

plt.figure(figsize=(7, 5))

plt.loglog(
    grid_sizes,
    [
        np.nan if np.isnan(
            grid_summary.loc[i, "Relative L2 change vs finer grid (%)"]
        )
        else grid_summary.loc[i, "Relative L2 change vs finer grid (%)"]
        for i in range(len(grid_sizes))
    ],
    marker="o",
    linewidth=1.8
)

plt.xlabel("Grid spacing, Δx = Δy (m)")
plt.ylabel("Relative L2 difference (%)")
plt.title("Spatial Grid-Refinement Convergence")
plt.grid(True, which="both", alpha=0.3)
plt.tight_layout()
plt.show()

# ==========================================================
# 16. MAXIMUM CONCENTRATION VS GRID SPACING
# ==========================================================

plt.figure(figsize=(7, 5))

plt.plot(
    grid_summary["Grid spacing (m)"],
    grid_summary["Maximum concentration"],
    marker="o",
    linewidth=1.8
)

plt.xlabel("Grid spacing, Δx = Δy (m)")
plt.ylabel("Maximum concentration")
plt.title("Maximum Concentration Under Grid Refinement")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# ==========================================================
# 17. SPATIAL CONCENTRATION FIELDS AT t = 120 s
# ==========================================================

fig, axes = plt.subplots(
    1,
    3,
    figsize=(15, 4.8)
)

fields = [
    (grid_results[0]["result"], "Δx = Δy = 4 m"),
    (grid_results[1]["result"], "Δx = Δy = 2 m"),
    (grid_results[2]["result"], "Δx = Δy = 1 m")
]

# Each item in `fields` is a tuple:
#     (result_dictionary, figure_title)
# Therefore access the first element before indexing the result dictionary.
vmax = max(
    np.max(result["final_field"])
    for result, title in fields
)

for ax, (result, title) in zip(axes, fields):

    contour = ax.contourf(
        result["X"],
        result["Y"],
        result["final_field"],
        levels=40,
        vmin=0,
        vmax=vmax
    )

    ax.scatter(
        xf,
        yf,
        s=45,
        marker="*"
    )

    ax.set_xlabel("x (m)")
    ax.set_ylabel("y (m)")
    ax.set_title(title)

fig.colorbar(
    contour,
    ax=axes.ravel().tolist(),
    label="Microplastic concentration"
)

fig.suptitle(
    "Spatial Grid Refinement at t = 120 s",
    y=1.02
)

plt.tight_layout()
plt.show()

# ==========================================================
# 18. FINAL NUMERICAL-ASSESSMENT SUMMARY
# ==========================================================

print("\n" + "=" * 72)
print("FINAL NUMERICAL-ASSESSMENT SUMMARY")
print("=" * 72)

print("\nA. Temporal stabilization")
if stabilization_time is not None:
    print(
        f"Sustained field-change criterion of "
        f"{temporal_tolerance:.2f}% was first satisfied at "
        f"approximately {stabilization_time:.2f} s."
    )
else:
    print(
        f"The {temporal_tolerance:.2f}% sustained field-change "
        "criterion was not reached within the tested period."
    )

print("\nB. Spatial grid refinement")
print(
    f"Relative L2 difference, 4 m -> 2 m: "
    f"{100.0 * E_4_to_2:.6f}%"
)
print(
    f"Relative L2 difference, 2 m -> 1 m: "
    f"{100.0 * E_2_to_1:.6f}%"
)

print(
    "\nInterpretation:"
)
print(
    "The temporal test evaluates stabilization of the transient "
    "concentration field, whereas the grid-refinement test evaluates "
    "sensitivity to spatial discretization."
)
print(
    "The 2 m grid is the production-grid resolution used in the main "
    "model configuration. The 4/2/1 m refinement study provides a "
    "separate numerical check on spatial discretization."
)
print(
    f"The grid-refinement comparison uses one common dt = {dt_grid:.6f} s, "
    "computed from the finest 1 m grid with a safety factor, so the "
    "comparison is not confounded by different time steps."
)
print(
    "The production simulation remains documented with dt = 0.25 s."
)
print(
    "The same homogeneous Neumann boundary treatment is used throughout "
    "the temporal and spatial numerical tests."
)
print("=" * 72)

# ==========================================================
# END — RUN THIS ENTIRE CELL ONCE
# ==========================================================
